# DO NOT USE - superseded by `panvram_lane0_A100_v3.ipynb`

Path 2 failed with `AttributeError: module 'panvram' has no attribute 'Cohort'`: the subprocess started in
`/content`, where the clone directory `/content/panvram` sits, so Python imported that directory as a namespace
package instead of the installed module. The same shadowing is the likely reason path 1's `ncu` runs returned exit
code 1. v3 runs from `/tmp` with `python -P`, prints `panvram.__file__` first and prints `ncu`'s stderr in full.


# panvram GPU decoder: how long is lane 0 alone? (v2)

`csrc/pv_cuda.cu` runs **one warp per refrel3 block**: lane 0 decodes the block's rANS stream and pushes copy
operations into a ring in shared memory, lanes 1..31 take them and copy the bases (`csrc/pv_cuda.cu`, the comment at
the top of the file and the body of `pv_queue_kernel`). While lane 0 is inside `r3_decode_stream`, the other 31 lanes have
nothing to do but poll `head`. This notebook measures what share of the block's time that is.

Lane 0 and lanes 1..31 are **the same warp**, so this is intra-warp divergence, not two warps competing. That rules
out warp-level occupancy metrics and points at two usable measurements:

1. **Without touching the code** - Nsight Compute: the average number of active threads per executed instruction. If
   the decode phase dominated the block, the ratio would sit near 1/32 = 0.031; if the copy phase dominated, near
   31/32 = 0.969. Plus the stall reasons, for context.
2. **With an instrumented build** - `clock64()` inside the kernel: lane 0 accumulates the cycles it spends in
   `r3_decode_stream`, lane 1 accumulates the cycles the consumers spend actually copying, and the difference against
   its total loop time is the waiting.

**Both paths can fail for reasons that have nothing to do with panvram**: Nsight Compute needs profiling permission,
which a hosted runtime may refuse (`ERR_NVGPUCTRPERM`), and the patch of path 2 was written on a host **with no GPU**,
so it has never been compiled. Each cell says what to do if its path dies; a result from either path is worth having,
and if they disagree that is itself the finding.

Protocol as in `logs/loader-2026-10-22`: 5 assemblies, batches of 1 024 windows, W = 8 192 and 32 768. The five
assemblies are the first five of the record's tar (release-2 haplotypes), not the five of the CPU rows.

In [ ]:
N_ASM = 5
BATCH = 1024
WS = (8192, 32768)
SEED = 20261003

print("=" * 46)
print(f"====   lane-0 study, N_ASM = {N_ASM}   ".ljust(46, "="))
print("=" * 46)
!nvidia-smi --query-gpu=name,memory.total,driver_version,clocks.max.sm --format=csv,noheader
!nproc

In [ ]:
!apt-get -qq install -y libzstd-dev > /dev/null
!git clone -q https://github.com/yasha1971-coder/panvram.git /content/panvram || true
%cd /content/panvram
!git log --oneline -1
import subprocess
print(subprocess.run(['git', 'status', '--short'], capture_output=True, text=True).stdout or '(clean tree)')

In [ ]:
# SPDX-License-Identifier: MIT
"""Fetch and verify a panvram q16k cohort from the Zenodo record, for the window-loader measurement.

The record (10.5281/zenodo.23232317) ships the 558 archives as one tar cut into 500 MiB parts; the individual
`*.q16k.rr3` files are NOT addressable on their own (they return 404). So:

  n_asm == 558: every part, checked against PARTS.sha256, joined, the whole tar checked against its .sha256,
                extracted, every archive checked against SHA256SUMS -> 558 archives.
  n_asm == 5:   the first 100 MB of part00 by HTTP range, the tar headers walked by hand, the archives that are
                complete inside that prefix extracted, the first five kept and checked against SHA256SUMS.q16k.

Either way: an archive of 0 bytes, a missing archive or a wrong SHA-256 raises RuntimeError at once.

Note for the five-assembly mode: the tar is ordered `./`, `./SHA256SUMS`, then the archives by name, so the five are
the first five of the release-2 assemblies (r2_HG00097_hap1, ...), NOT the five year-1 haplotypes of the CPU rows of
logs/loader-2026-10-22. The cohort size is what this mode controls; the names differ, and the report must say so.
"""
import hashlib
import os
import shutil
import subprocess
import tarfile
from pathlib import Path

REC = "https://zenodo.org/records/23232317/files"
PREFIX_BYTES = 100 * 1024 * 1024
BLOCK = 512


def _md5(path, buf=8 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            b = f.read(buf)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def _sha256(path, buf=8 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(buf)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def _fetch(url, dest, byte_range=None, tries=5):
    dest = Path(dest)
    for attempt in range(1, tries + 1):
        cmd = ["curl", "-fsS", "--retry", "3", "--retry-delay", "5", "-o", str(dest)]
        if byte_range:
            cmd += ["-r", byte_range]
        r = subprocess.run(cmd + [url], capture_output=True, text=True)
        if r.returncode == 0 and dest.exists() and dest.stat().st_size > 0:
            return dest
        if attempt == tries:
            raise RuntimeError(f"download failed after {tries} attempts: {url}\n"
                               f"curl said: {r.stderr.strip() or r.returncode}; "
                               f"size on disk: {dest.stat().st_size if dest.exists() else 'no file'}")
    raise AssertionError


def _read_sums(path):
    sums = {}
    for line in open(path):
        line = line.strip()
        if not line:
            continue
        digest, name = line.split(None, 1)
        sums[os.path.basename(name.strip())] = digest
    return sums


def _verify(dirpath, sums, expect_n):
    dirpath = Path(dirpath)
    got = sorted(p for p in dirpath.glob("*.q16k.rr3"))
    if len(got) != expect_n:
        raise RuntimeError(f"expected {expect_n} archives in {dirpath}, found {len(got)}")
    for p in got:
        if p.stat().st_size == 0:
            raise RuntimeError(f"{p.name} is 0 bytes - the download did not deliver the file")
        want = sums.get(p.name)
        if want is None:
            raise RuntimeError(f"{p.name} is not listed in the checksum file")
        have = _sha256(p)
        if have != want:
            raise RuntimeError(f"{p.name}: SHA-256 {have} does not match the record's {want}")
    print(f"VERIFIED\t{len(got)} archives, every SHA-256 equals the record's, no empty file")
    for p in got[:3]:
        print(f"  {p.name}\t{p.stat().st_size} B\tsha256 {_sha256(p)}\tmd5 {_md5(p)}")
    if len(got) > 3:
        print(f"  ... {len(got) - 3} more")
    return got


def _walk_tar_prefix(buf_path, out_dir, want):
    """Extract the archives wholly contained in a truncated tar, in tar order, up to `want` of them."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    taken, members = [], []
    with open(buf_path, "rb") as f:
        size = Path(buf_path).stat().st_size
        off = 0
        while off + BLOCK <= size and len(taken) < want:
            f.seek(off)
            head = f.read(BLOCK)
            if head[:1] == b"\0":
                break
            name = head[0:100].rstrip(b"\0").decode("utf-8", "replace")
            raw = head[124:136].rstrip(b"\0 ").decode() or "0"
            msize = int(raw, 8)
            data_off = off + BLOCK
            padded = (msize + BLOCK - 1) // BLOCK * BLOCK
            members.append((name, msize))
            base = os.path.basename(name)
            if base.endswith(".q16k.rr3"):
                if data_off + msize > size:
                    break                                    # truncated member: stop, do not write a short file
                f.seek(data_off)
                (out_dir / base).write_bytes(f.read(msize))
                taken.append(base)
            elif base == "SHA256SUMS" and data_off + msize <= size:
                f.seek(data_off)
                (out_dir / "SHA256SUMS.in_tar").write_bytes(f.read(msize))
            off = data_off + padded
    print(f"PREFIX\t{len(members)} tar members seen, {len(taken)} archives complete: {', '.join(taken)}")
    if len(taken) < want:
        raise RuntimeError(f"only {len(taken)} complete archives in the first {PREFIX_BYTES} bytes, need {want}; "
                           f"raise PREFIX_BYTES")
    return taken


def prepare_cohort(n_asm, dest="cohort", work="dl"):
    """Put a verified q16k cohort of n_asm assemblies in `dest`. Returns the list of archive paths."""
    if n_asm not in (5, 558):
        raise ValueError("n_asm must be 5 or 558")
    dest, work = Path(dest), Path(work)
    dest.mkdir(parents=True, exist_ok=True)
    work.mkdir(parents=True, exist_ok=True)

    if n_asm == 558:
        parts = [f"cohort558_q16k.tar.part{i:02d}" for i in range(16)]
        _fetch(f"{REC}/PARTS.sha256", work / "PARTS.sha256")
        psums = _read_sums(work / "PARTS.sha256")
        for p in parts:
            q = work / p
            if not (q.exists() and q.stat().st_size and _sha256(q) == psums.get(p)):
                _fetch(f"{REC}/{p}", q)
                have = _sha256(q)
                if have != psums.get(p):
                    raise RuntimeError(f"{p}: SHA-256 {have} does not match PARTS.sha256 {psums.get(p)}")
            print(f"  part ok {p} {q.stat().st_size} B", flush=True)
        tar = work / "cohort558_q16k.tar"
        with open(tar, "wb") as out:
            for p in parts:
                with open(work / p, "rb") as f:
                    shutil.copyfileobj(f, out, 8 << 20)
        _fetch(f"{REC}/cohort558_q16k.tar.sha256", work / "tar.sha256")
        want_tar = open(work / "tar.sha256").read().split()[0]
        have_tar = _sha256(tar)
        if have_tar != want_tar:
            raise RuntimeError(f"joined tar SHA-256 {have_tar} does not match the record's {want_tar}")
        print(f"TAR\t{tar.stat().st_size} B, SHA-256 matches the record", flush=True)
        with tarfile.open(tar) as t:
            t.extractall(dest, filter="data")
        for p in parts:
            (work / p).unlink(missing_ok=True)
        tar.unlink(missing_ok=True)
        inner = dest / "SHA256SUMS"
        if not inner.exists():
            raise RuntimeError("the tar did not contain SHA256SUMS")
        return _verify(dest, _read_sums(inner), 558)

    prefix = work / "part00.prefix"
    _fetch(f"{REC}/cohort558_q16k.tar.part00", prefix, byte_range=f"0-{PREFIX_BYTES - 1}")
    print(f"RANGE\t{prefix.stat().st_size} B of part00", flush=True)
    _walk_tar_prefix(prefix, dest, want=5)
    _fetch(f"{REC}/SHA256SUMS.q16k", work / "SHA256SUMS.q16k")
    out = _verify(dest, _read_sums(work / "SHA256SUMS.q16k"), 5)
    prefix.unlink(missing_ok=True)
    return out


In [ ]:
# Reference. What the archives actually bind is the DECODED reference: the bases of the FASTA, upper-cased,
# SHA-256 a6e4a745... (DATASET.md). That is header-independent, so two different FASTA renderings of
# T2T-CHM13v2.0 both pass it. We fetch the file DATASET.md pins (NCBI GCA_009914755.4) and report every sum.
import hashlib, os
from pathlib import Path
%cd /content
Path('cohort').mkdir(exist_ok=True)
REF_URL = ('https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/009/914/755/'
           'GCA_009914755.4_T2T-CHM13v2.0/GCA_009914755.4_T2T-CHM13v2.0_genomic.fna.gz')
PINNED_BYTES = 3156259565
PINNED_SHA256 = '51ab7d3c2a375ecff53ee2a1be03e58d57cc1601c948a860c03a78d724f26e4c'
PINNED_MD5 = 'cd1e52ce400c027ed0b7ab4b9d613f5a'          # the unpacked NCBI FASTA; md5 as printed in DATASET.md
DECODED_SHA256 = 'a6e4a74516252f28ac61291c9e11eba9e9cd53a576e014b3dfd51e4816333080'   # its bases, upper-cased

def _md5(path, buf=8 << 20):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        while True:
            b = f.read(buf)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def _decoded_sha256(path):
    """SHA-256 of the bases only, upper-cased: what open_v1 checks against every archive header."""
    h = hashlib.sha256()
    drop = bytes.maketrans(b'acgtn', b'ACGTN')
    with open(path, 'rb') as f:
        for line in f:
            if line[:1] != b'>':
                h.update(line.translate(drop, b'\r\n'))
    return h.hexdigest()

ref = Path('cohort/reference.fa')
if not ref.exists():
    !curl -fsSL -o t2t.fna.gz "$REF_URL"
    !gunzip -f t2t.fna.gz && mv t2t.fna cohort/reference.fa
n, h256, hmd5 = ref.stat().st_size, _sha256(ref), _md5(ref)
print(f'reference file  {n} B')
print(f'  sha256 {h256}')
print(f'  md5    {hmd5}')
print(f'  pinned {PINNED_BYTES} B, sha256 {PINNED_SHA256}, md5 {PINNED_MD5}')
same = (n, h256, hmd5) == (PINNED_BYTES, PINNED_SHA256, PINNED_MD5)
print('  this file is', 'the pinned NCBI FASTA' if same else 'a DIFFERENT rendering - the bases decide')
dec = _decoded_sha256(ref)
print(f'decoded reference (bases, upper-cased)')
print(f'  sha256   {dec}')
print(f'  expected {DECODED_SHA256}')
if dec != DECODED_SHA256:
    raise SystemExit('this is not T2T-CHM13v2.0 as the archives were built against - they would refuse it')
print('reference OK: the decoded bases are what the archives bind')


In [ ]:
paths = prepare_cohort(N_ASM, dest='/content/cohort', work='/content/dl')
print(f'{len(paths)} archives ready')
!du -sh /content/cohort

## Path 1: Nsight Compute, no code change

`smsp__thread_inst_executed_per_inst_executed.ratio` is the average number of active threads per executed
instruction for the kernel. Read it against the two ends: 1/32 = 0.031 means the warp is almost always running one
lane, 31/32 = 0.969 means almost always 31.

This profiles the **shipped** kernel, so whatever it reports is a property of the published code.

In [ ]:
# build the shipped extension (no patch) and profile one batch
%cd /content/panvram
!git checkout -- csrc/ 2>/dev/null || true
!pip install -q -e . 2>&1 | tail -2
import panvram, torch
assert panvram.with_cuda, 'the CUDA extension did not build'
print('panvram', getattr(panvram, '__version__', 'ok'), '| torch', torch.__version__, '|', torch.cuda.get_device_name(0))

prof = '''import torch, panvram, sys
W = int(sys.argv[1]); BATCH = int(sys.argv[2])
c = panvram.Cohort.open('/content/cohort', device='cuda', dataset='q16k')
g = torch.Generator(device='cuda').manual_seed(20261003)
c.sample(BATCH, W, generator=g); torch.cuda.synchronize()
x = c.sample(BATCH, W, generator=g); torch.cuda.synchronize()
print('ok', tuple(x.shape))
'''
open('/content/prof_one.py', 'w').write(prof)

!which ncu || (apt-get -qq install -y nsight-compute-2024.1.0 > /dev/null 2>&1; true)
!which ncu || ls /opt/nvidia/nsight-compute/*/ncu 2>/dev/null | head -1

In [ ]:
import shutil, subprocess
NCU = shutil.which('ncu') or next(iter(__import__('glob').glob('/opt/nvidia/nsight-compute/*/ncu')), None)
METRICS = ','.join(['smsp__thread_inst_executed_per_inst_executed.ratio',
                    'smsp__inst_executed.sum',
                    'smsp__warp_issue_stalled_long_scoreboard_per_warp_active.pct',
                    'smsp__warp_issue_stalled_barrier_per_warp_active.pct',
                    'smsp__warp_issue_stalled_short_scoreboard_per_warp_active.pct'])
if NCU is None:
    print('NOT MEASURED: Nsight Compute is not installed in this runtime; go to path 2')
else:
    for W in WS:
        print(f'=== W = {W}')
        r = subprocess.run([NCU, '--kernel-name', 'regex:pv_queue_kernel', '--launch-count', '1',
                            '--metrics', METRICS, '--csv', 'python3', '/content/prof_one.py', str(W), str(BATCH)],
                           capture_output=True, text=True)
        out = r.stdout.strip()
        if 'ERR_NVGPUCTRPERM' in r.stdout + r.stderr or not out:
            print('NOT MEASURED: this runtime refuses GPU performance counters (ERR_NVGPUCTRPERM) - go to path 2')
            print((r.stderr or r.stdout)[-400:])
            break
        print(out[-2500:])

## Path 2: an instrumented build, `clock64()`

The patch below adds three counters to `pv_queue_kernel` and one function to read them. It changes **no decode logic**:
lane 0 times its own call to `r3_decode_stream`, lane 1 times the stretches where it is actually copying and its whole
loop, and the difference is the waiting.

```
cyc[0] += lane 0's cycles inside r3_decode_stream
cyc[1] += lane 1's cycles spent executing queue entries
cyc[2] += lane 1's cycles from entering the loop to leaving it
cyc[3] += 1 per block
```

Then: `waiting share = (cyc[2] - cyc[1]) / cyc[2]`, and `lane-0 share of the block = cyc[0] / cyc[2]`.

**This patch has never been compiled** - it was written on a host without `nvcc`. Every anchor is asserted before
anything is written, so a changed upstream file stops the cell instead of corrupting it. If the build fails, report
path 1's number and say path 2 did not build.

In [ ]:
import re
from pathlib import Path
cu = Path('/content/panvram/csrc/pv_cuda.cu'); core = Path('/content/panvram/csrc/pv_core.cpp')
s, c = cu.read_text(), core.read_text()

A1 = 'const uint32_t lane = threadIdx.x;'
A2 = '        s_nk = r3_decode_stream(src, slen, D.T + a, ref, D.ref_n, blen, g0, sk);'
A3 = '        const unsigned m = 0xFFFFFFFEu; int next = 0; uint32_t spin = 0;'
A4 = '''                __syncwarp(m); next++; if (lane == 1) { __threadfence_block(); *tail = next; } spin = 0;'''
A5 = '            if (next < h) {'
A6 = '            } else if (d) { if (next >= h) break; }'
for i, a in enumerate((A1, A2, A3, A4, A5, A6), 1):
    assert s.count(a) == 1, f'anchor A{i} found {s.count(a)} times - upstream changed, stopping'

A0 = 'template <bool PACKED, bool COMPACT>'
assert s.count(A0) == 1, f'anchor A0 found {s.count(A0)} times - upstream changed, stopping'
s = s.replace(A0, '__device__ unsigned long long pv_cyc[4];\n\n' + A0, 1)
assert 'pv_cyc[4]' in s
s = s.replace(A2, '        const unsigned long long _t0 = clock64();\n' + A2 +
              '\n        atomicAdd(&pv_cyc[0], clock64() - _t0); atomicAdd(&pv_cyc[3], 1ull);', 1)
s = s.replace(A3, A3 + '\n        unsigned long long _cwork = 0; const unsigned long long _cl0 = clock64();', 1)
s = s.replace(A5, A5 + '\n                const unsigned long long _w0 = clock64();', 1)
s = s.replace(A4, '                if (lane == 1) _cwork += clock64() - _w0;\n' + A4, 1)
s = s.replace(A6, A6.replace('break; }', 'break; }') +
              '\n            // accounting below on leaving the loop', 1)
s = s.replace('''        }
    }
    __syncwarp();
    if (s_nk < 0)''', '''        }
        if (lane == 1) { atomicAdd(&pv_cyc[1], _cwork); atomicAdd(&pv_cyc[2], clock64() - _cl0); }
    }
    __syncwarp();
    if (s_nk < 0)''', 1)

READER = '''
#include <vector>
std::vector<unsigned long long> pv_read_cycles(bool reset) {
    unsigned long long h[4] = {0, 0, 0, 0};
    cudaMemcpyFromSymbol(h, pv_cyc, sizeof(h));
    if (reset) { unsigned long long z[4] = {0, 0, 0, 0}; cudaMemcpyToSymbol(pv_cyc, z, sizeof(z)); }
    return std::vector<unsigned long long>(h, h + 4);
}
'''
s = s + READER
assert 'PYBIND11_MODULE' in c or 'm.def(' in c, 'pv_core.cpp does not look like the pybind module'
hook = re.search(r'(PYBIND11_MODULE\s*\([^)]*\)\s*\{)', c)
assert hook, 'could not find the pybind module block'
c = (c[:hook.end()] +
     '\n#ifdef PV_WITH_CUDA\n    { extern std::vector<unsigned long long> pv_read_cycles(bool);'
     '\n      m.def("read_cycles", &pv_read_cycles); }\n#endif\n' +
     c[hook.end():])
cu.write_text(s); core.write_text(c)
print('patched; diff summary:')
!cd /content/panvram && git diff --stat -- csrc/
!cd /content/panvram && git diff -- csrc/pv_cuda.cu | head -60

In [ ]:
# Path 2 must run in a FRESH interpreter. v1 of this notebook failed here with "read_cycles present: False":
# path 1 had already imported panvram, so the old panvram/_C*.so was loaded into this kernel, and a native
# extension cannot be swapped by importlib.reload - the rebuilt module is simply never seen. So the instrumented
# build is driven, and the measurement run, through subprocess.
import json, subprocess, sys, textwrap
from pathlib import Path

script = textwrap.dedent('''
    import json, sys, time
    import torch, panvram
    from panvram import _C
    if not hasattr(_C, "read_cycles"):
        print("NO_READ_CYCLES", file=sys.stderr); sys.exit(3)
    BATCH, WS, SEED, RUNS = {batch}, {ws}, {seed}, 5
    c = panvram.Cohort.open("/content/cohort", device="cuda", dataset="q16k")
    out = {{"gpu": torch.cuda.get_device_name(0), "assemblies": len(c.names), "batch": BATCH, "rows": []}}
    for W in WS:
        g = torch.Generator(device="cuda").manual_seed(SEED)
        for _ in range(3):
            c.sample(BATCH, W, generator=g)
        torch.cuda.synchronize()
        _C.read_cycles(True)
        g = torch.Generator(device="cuda").manual_seed(SEED)
        t0 = time.perf_counter(); x = c.sample(BATCH, W, generator=g); torch.cuda.synchronize()
        wall = time.perf_counter() - t0
        d0, work, loop, blocks = _C.read_cycles(True)
        print(f"=== W = {{W}}: {{blocks}} blocks, wall {{wall*1e3:.3f}} ms")
        if not blocks or not loop:
            print("  NOT MEASURED: counters stayed at zero"); continue
        print(f"  lane 0 in r3_decode_stream     : {{d0/blocks:12.0f}} cycles per block")
        print(f"  lanes 1..31 copying            : {{work/blocks:12.0f}}")
        print(f"  lanes 1..31 loop total         : {{loop/blocks:12.0f}}")
        print(f"  waiting share of the consumers : {{100*(loop-work)/loop:6.2f}} %")
        print(f"  lane 0 decode / consumer loop  : {{100*d0/loop:6.2f}} %")
        out["rows"].append({{"W": W, "blocks": blocks, "wall_s": wall, "cyc_decode": d0,
                            "cyc_copy": work, "cyc_loop": loop,
                            "wait_share_pct": 100*(loop-work)/loop, "lane0_share_pct": 100*d0/loop}})
    # the instrumented kernel must still decode correctly
    W = WS[0]
    g = torch.Generator(device="cuda").manual_seed(SEED)
    x, coords = c.sample(256, W, generator=g, return_coords=True)
    cc = panvram.Cohort.open("/content/cohort", device="cpu", dataset="q16k")
    bad = 0
    for i in range(coords.shape[0]):
        a, ci, s, _ = [int(v) for v in coords[i]]
        if not torch.equal(x[i].cpu(), cc.fetch(cc.names[a], cc._core.contigs(a)[ci][0], s, W)):
            bad += 1
    print(f"CHECK\\tW {{W}}\\twindows {{coords.shape[0]}}\\tidentical {{coords.shape[0]-bad}}\\tdiffer {{bad}}")
    out["check_differ"] = bad
    json.dump(out, open("/content/lane0_result.json", "w"), indent=1)
    print("written /content/lane0_result.json")
''').format(batch=BATCH, ws=tuple(WS), seed=SEED)
Path('/content/lane0_run.py').write_text(script)

print('--- rebuilding the patched extension in a clean subprocess')
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], cwd='/content/panvram',
                   capture_output=True, text=True)
print((r.stdout + r.stderr)[-800:] or 'build ok')
if r.returncode:
    raise SystemExit('the instrumented build failed to compile - report path 1 only, and paste the error above')

print('--- running path 2 in a fresh interpreter')
r = subprocess.run([sys.executable, '/content/lane0_run.py'], capture_output=True, text=True)
print(r.stdout[-4000:])
if r.returncode:
    print(r.stderr[-2000:])
    raise SystemExit('path 2 did not produce numbers - report path 1 only')
